In [ ]:
from pathlib import Path
import geopandas as gpd
import pandas as pd
from shapely.validation import explain_validity
from IPython.display import display

### Read in files

In [ ]:
base_path = Path("/Users/robynhaggis/Documents/Geospatial_analysis/dphil_papers")
output_dir = base_path / "dphil_paper_2/results"

In [ ]:
land_use = gpd.read_file(base_path / "dphil_common_cross_cutting/common_incoming_data/landcover/2013_landcover/2013_landuse_LandCover.shp")
print(land_use.crs)

In [ ]:
admin_boundary_path = base_path / "dphil_common_cross_cutting/common_incoming_data/boundaries/admin_boundaries.gpkg"
admin1 = gpd.read_file(admin_boundary_path, layer="admin1")
admin1.crs
admin1.head()

In [ ]:
catchments_unionized_final = base_path / "dphil_paper_2/processed_data/major_river_catchments/major_basins_plus_coastal_unionized_final.gpkg"
catchments = gpd.read_file(catchments_unionized_final)
catchments.head()
print("Catchments:", len(catchments))

### Analyse Parish information in relation to catchments

In [ ]:
# Intersections of catchments with Admin-1 (PARISH)
intersections = gpd.overlay(
    catchments[["catchment_uid", "geometry"]],
    admin1[["PARISH", "geometry"]],
    how="intersection"
).copy()

# Areas
intersections["part_m2"] = intersections.geometry.area

catchments["area_km2"] = catchments.geometry.area / 1e6  # assumes current CRS is metres
catchment_area_m2 = catchments.set_index("catchment_uid").geometry.area

# Summarize shares per (catchment, PARISH)
grp = (intersections
       .groupby(["catchment_uid", "PARISH"], as_index=False)
       .agg(part_m2=("part_m2", "sum")))

grp["catchment_area_m2"] = grp["catchment_uid"].map(catchment_area_m2)
shares = grp.assign(
    area_km2 = grp["part_m2"] / 1e6,
    share    = grp["part_m2"] / grp["catchment_area_m2"],
    share_pct = lambda d: (d["share"] * 100).round(1).clip(0, 100)
).drop(columns="part_m2")

intersections

In [ ]:
# === Parish breakdowns for *every* catchment =================================
# 1) Compact per-catchment breakdown string (e.g., "ParishA (62.3%), ParishB (37.7%)")
catchment_administrative_share = shares.copy()  # use shares_nosliver if you filtered tiny slivers
catchment_administrative_share["share_pct"] = catchment_administrative_share["share_pct"].round(1)

brk_all = (catchment_administrative_share
           .sort_values(["catchment_uid", "share"], ascending=[True, False])
           .assign(entry=lambda d: d["PARISH"] + " (" + d["share_pct"].astype(str) + "%)")
           .groupby("catchment_uid", as_index=False)["entry"]
           .agg(", ".join)
           .rename(columns={"entry": "parish_breakdown"}))

# 2) Majority parish (and add area if you have it)
maj = (catchment_administrative_share.loc[catchment_administrative_share.groupby("catchment_uid")["share"].idxmax(),
                          ["catchment_uid", "PARISH", "share_pct"]]
       .rename(columns={"PARISH": "majority_parish",
                        "share_pct": "majority_share_pct"}))

areas = (catchments[["catchment_uid","area_km2"]]
         if "area_km2" in catchments.columns
         else catchments[["catchment_uid"]].assign(area_km2=catchments.geometry.area/1e6))

out = (brk_all
       .merge(maj, on="catchment_uid", how="left")
       .merge(areas, on="catchment_uid", how="left")
       .sort_values("catchment_uid"))

# 3) Save CSVs
out_csv = output_dir / "catchment_attributes/catchment_parish_breakdown_all.csv"
out.to_csv(out_csv, index=False)
print("Saved:", out_csv)

In [ ]:
sizes = catchments["area_km2"]  # or out["area_km2"] — they’re the same values

min_km2  = sizes.min()
mean_km2 = sizes.mean()
max_km2  = sizes.max()
range_km2 = max_km2 - min_km2

print(f"Count: {sizes.size}")
print(f"Min (km²):   {min_km2:.2f}")
print(f"Mean (km²):  {mean_km2:.2f}")
print(f"Max (km²):   {max_km2:.2f}")
print(f"Range (km²): {range_km2:.2f}")

# Optional: save a tidy one-row table
stats_df = pd.DataFrame([{
    "count": sizes.size,
    "min_km2": min_km2,
    "mean_km2": mean_km2,
    "max_km2": max_km2,
    "range_km2": range_km2,
}])
stats_df.to_csv(output_dir / "catchment_attributes/catchment_size_stats.csv", index=False)

In [ ]:
catchments = catchments.copy()
catchments["area_m2"]  = catchments.geometry.area
catchments["area_km2"] = catchments["area_m2"] / 1e6

# Identify the smallest catchment
smallest_catchment = catchments["area_km2"].idxmin()
row   = catchments.loc[smallest_catchment, ["catchment_uid", "area_m2", "area_km2"]]
geom  = catchments.geometry.iloc[smallest_catchment]

print("Smallest catchment record:")
print(row.to_string())
print(f"Geom type: {geom.geom_type} | valid: {geom.is_valid} | empty: {geom.is_empty}")
print("Validity detail:", explain_validity(geom))
print("Bounds (xmin, ymin, xmax, ymax):", geom.bounds)

# Show top 5 smallest by area (km²) with more precision
print("\nTop 30 smallest catchments (km², 8 d.p.):")
print(
    catchments[["catchment_uid","area_km2"]]
      .sort_values("area_km2")
      .head(30)
      .assign(area_km2=lambda d: d["area_km2"].map(lambda x: f"{x:.8f}"))
      .to_string(index=False)
)

# If it's invalid or reported as zero area, try a simple fix and report the fixed area
if (not geom.is_valid) or (row["area_m2"] == 0):
    fixed = geom.buffer(0)  # simple make-valid trick
    area_fixed_m2 = fixed.area
    print(f"\nAfter buffer(0): {area_fixed_m2/1e6:.8f} km²  ({area_fixed_m2:,.6f} m²)")

### FOREST CATEGORIES

In [ ]:
# --- 1) Build class→fraction mapping for categories --------------------------
CAT_EXIST = "existing_forest"
CAT_REFO  = "reforestable"
CAT_TREAT = "treated_as_forest"
CAT_OTHER = "other"

# Union of all labels we know about
all_labels = (
    set(existing_forest_classes)
    | set(reforestable_classes)
    | set(treated_as_forest_classes)
    | set(mixed_land_use_fractions.keys())
)

# Detect the land-use class column by which column matches the most labels
str_cols = [c for c in land_use.columns if land_use[c].dtype == object]
hits = {c: land_use[c].isin(all_labels).sum() for c in str_cols}
LU_COL = max(hits, key=hits.get) if hits else None
if LU_COL is None or hits[LU_COL] == 0:
    raise ValueError("Could not detect the land-use class column—set LU_COL to the correct field.")

# Map each class label to category fractions
class_to_frac = {lbl: {} for lbl in all_labels}
for lbl in existing_forest_classes:
    class_to_frac.setdefault(lbl, {})[CAT_EXIST] = 1.0
for lbl in reforestable_classes:
    class_to_frac.setdefault(lbl, {})[CAT_REFO] = 1.0
for lbl in treated_as_forest_classes:
    class_to_frac.setdefault(lbl, {})[CAT_TREAT] = 1.0
for lbl, parts in mixed_land_use_fractions.items():
    for cat_name, frac in parts.items():
        # normalize keys from your dict to our category names
        if "existing_forest" in cat_name:
            cat = CAT_EXIST
        elif "reforestable" in cat_name:
            cat = CAT_REFO
        elif "treated_as_forest" in cat_name or "treated" in cat_name:
            cat = CAT_TREAT
        else:
            cat = CAT_OTHER
        class_to_frac.setdefault(lbl, {})[cat] = float(frac)

# Any class in your data but not in our mapping becomes 'other'
present_labels = set(land_use[LU_COL].unique())
for lbl in present_labels:
    if lbl not in class_to_frac:
        class_to_frac[lbl] = {CAT_OTHER: 1.0}
    elif not class_to_frac[lbl]:
        class_to_frac[lbl] = {CAT_OTHER: 1.0}

In [ ]:
# Named category columns (for ordering and completeness)
category_cols = [CAT_EXIST, CAT_REFO, CAT_TREAT, CAT_OTHER]

# Ensure CRS match for overlay
if land_use.crs != catchments.crs:
    land_use = land_use.to_crs(catchments.crs)
    
# 2) Intersect land-use with catchments and compute piece areas (m²)
landuse_catchment_intersections = gpd.overlay(
    land_use[[LU_COL, "geometry"]],
    catchments[["catchment_uid", "geometry"]],
    how="intersection"
).rename(columns={LU_COL: "lu_class"})
landuse_catchment_intersections["area_m2"] = landuse_catchment_intersections.geometry.area

In [ ]:
# 3) Allocate each piece to categories (handles fractional 'mixed' classes)
rows = []
append = rows.append
for cls, uid, a in zip(
    landuse_catchment_intersections["lu_class"].values,
    landuse_catchment_intersections["catchment_uid"].values,
    landuse_catchment_intersections["area_m2"].values
):
    fracs = class_to_frac.get(cls, {CAT_OTHER: 1.0})
    for category_name, frac in fracs.items():
        if frac:
            append((uid, category_name, float(a) * float(frac)))

category_area_allocations = pd.DataFrame(
    rows, columns=["catchment_uid", "category", "area_m2"]
)
if category_area_allocations.empty:
    raise RuntimeError("No allocations produced — check LU_COL and class names.")

In [ ]:
# 4) Aggregate to one row per catchment and build clearly labelled outputs
catchment_category_area_m2_long = (
    category_area_allocations
    .groupby(["catchment_uid", "category"], as_index=False)
    .agg(area_m2=("area_m2", "sum"))
)

# Pivot to wide by category, ensure all expected columns exist and ordered
catchment_category_area_m2_by_catchment = (
    catchment_category_area_m2_long
    .pivot(index="catchment_uid", columns="category", values="area_m2")
    .reindex(columns=category_cols, fill_value=0.0)
)

# --- m² per category (suffix _m2) ---
cat_area_m2 = catchment_category_area_m2_by_catchment.copy()
cat_area_m2.columns = [f"{c}_m2" for c in cat_area_m2.columns]

# Totals of mapped land-use inside each catchment
total_m2 = cat_area_m2.sum(axis=1)

# --- km² per category (suffix _km2) ---
cat_area_km2 = cat_area_m2 / 1e6
cat_area_km2.columns = [c.replace("_m2", "_km2") for c in cat_area_km2.columns]

# --- % of mapped land-use inside each catchment (suffix _pct) ---
cat_area_pct_of_mapped = cat_area_m2.div(total_m2.where(total_m2 > 0, pd.NA), axis=0) * 100
cat_area_pct_of_mapped.columns = [c.replace("_m2", "_pct") for c in cat_area_pct_of_mapped.columns]

# Final per-catchment stats table (mapped land-use only)
catchment_landuse_by_category_stats = (
    pd.concat([cat_area_m2, cat_area_km2, cat_area_pct_of_mapped], axis=1)
      .assign(total_m2=total_m2, total_km2=total_m2 / 1e6)
      .reset_index()
      .sort_values("catchment_uid")
)

In [ ]:
# 1) Identify which classes were mapped to 'other'
other_classes = {lbl for lbl, parts in class_to_frac.items() if parts.get(CAT_OTHER, 0.0) > 0}

# 2) Keep intersection pieces that belong to 'other' classes and aggregate
if other_classes:
    other_pieces = (
        landuse_catchment_intersections
        .loc[landuse_catchment_intersections["lu_class"].isin(other_classes),
             ["catchment_uid", "lu_class", "area_m2"]]
        .copy()
    )
else:
    other_pieces = pd.DataFrame(columns=["catchment_uid", "lu_class", "area_m2"])

other_by_class_long = (
    other_pieces
    .groupby(["catchment_uid", "lu_class"], as_index=False)
    .agg(area_m2=("area_m2", "sum"))
    .assign(area_km2=lambda d: d["area_m2"] / 1e6)
)

# 3) Percentages: of mapped land-use (always) and of the catchment polygon (if available)
#    'total_m2' was computed above from cat_area_m2.sum(axis=1), indexed by catchment_uid
other_by_class_long["pct_of_mapped"] = (
    other_by_class_long["area_m2"].div(
        other_by_class_long["catchment_uid"].map(total_m2).where(total_m2 > 0),
    ) * 100
)

has_catch_area = ("area_km2" in out.columns)
if has_catch_area:
    catch_area_m2 = (out.set_index("catchment_uid")["area_km2"] * 1e6)
    other_by_class_long["pct_of_catchment"] = (
        other_by_class_long["area_m2"].div(
            other_by_class_long["catchment_uid"].map(catch_area_m2).where(catch_area_m2 > 0),
        ) * 100
    )


In [ ]:
# 4) Human-readable summary strings per catchment (top classes; hide tiny slivers)
def _mk_summary(grp, metric_col, top=6, min_pct=0.05):
    s = grp.dropna(subset=[metric_col]).sort_values(metric_col, ascending=False)
    s = s[s[metric_col] >= min_pct]  # hide very tiny entries (<0.05%)
    if s.empty:
        return ""
    parts = [f"{cls} ({pct:.1f}%)" for cls, pct in zip(s["lu_class"], s[metric_col])]
    return "; ".join(parts[:top])

# Build Series to merge
sum_mapped = (
    other_by_class_long.groupby("catchment_uid")
    .apply(_mk_summary, metric_col="pct_of_mapped")
    .rename("other_breakdown_pct_of_mapped")
)

catchment_landuse_by_category_stats = (
    catchment_landuse_by_category_stats
    .merge(sum_mapped, on="catchment_uid", how="left")
)

if has_catch_area and "pct_of_catchment" in other_by_class_long.columns:
    sum_catch = (
        other_by_class_long.groupby("catchment_uid")
        .apply(_mk_summary, metric_col="pct_of_catchment")
        .rename("other_breakdown_pct_of_catchment")
    )
    catchment_landuse_by_category_stats = (
        catchment_landuse_by_category_stats
        .merge(sum_catch, on="catchment_uid", how="left")
    )

# Quick preview in your existing display (add the new column)
extra_cols = ["other_breakdown_pct_of_mapped"]
if has_catch_area:
    extra_cols.append("other_breakdown_pct_of_catchment")

# if has_catch_area and "other_breakdown_pct_of_catchment" in catchment_table_with_landuse.columns:
#     preview_cols.append("other_breakdown_pct_of_catchment")


# (Optional) Also compute % relative to the entire catchment polygon area
if "area_km2" in out.columns:
    area_m2_series = out.set_index("catchment_uid")["area_km2"] * 1e6
    tmp = catchment_landuse_by_category_stats.set_index("catchment_uid").copy()
    for base in category_cols:
        tmp[f"{base}_of_catchment_pct"] = (
            (tmp[f"{base}_m2"] / area_m2_series).where(area_m2_series > 0, pd.NA) * 100
        )
    catchment_landuse_by_category_stats = tmp.reset_index()

# 5) Merge into your existing summary table and save
catchment_table_with_landuse = out.merge(
    catchment_landuse_by_category_stats, on="catchment_uid", how="left"
)

csv_path = output_dir / "catchment_attributes/catchment_landuse_by_category_stats.csv"
catchment_table_with_landuse.to_csv(csv_path, index=False)
print("Saved:", csv_path)

# Readable preview (km² + % of mapped area)
pd.set_option("display.float_format", "{:,.2f}".format)

preview_cols = [
    "catchment_uid",
    "existing_forest_km2", "reforestable_km2", "treated_as_forest_km2", "other_km2",
    "total_km2",
    "existing_forest_pct", "reforestable_pct", "treated_as_forest_pct", "other_pct",
    "other_breakdown_pct_of_mapped",
]
display(catchment_table_with_landuse[preview_cols].head(12))